# ASR Lab quickstart

Run the evaluation pipeline of [multilingual-asr-lab](https://github.com/ivannamor/multilingual-asr-lab)
yourself: download a few utterances of FLEURS, transcribe them with Whisper, and measure word error
rate (WER) with a bootstrap confidence interval, the most frequent errors, and the effect of text
normalization.

- **On Google Colab** (recommended): *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.
- **Locally**: from a clone of the repository, `uv sync` then `uv run --with jupyter jupyter lab`.

The full results and their discussion are in
[RESULTS.md](https://github.com/ivannamor/multilingual-asr-lab/blob/main/RESULTS.md). This notebook
contains no logic of its own: every step calls the `asr_lab` package.

## 1. Install

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("asr_lab") is None:  # on Colab, or any environment without the package
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "asr-lab @ git+https://github.com/ivannamor/multilingual-asr-lab",
        ],
        check=True,
    )

## 2. Settings

Change these and run the following cells again. With a GPU, `large-v3-turbo` is a much stronger
model; on a CPU keep `small` or smaller. FLEURS languages available here: `it`, `de`, `en`.

In [ ]:
import ctranslate2

LANG = "it"  # "it", "de" or "en"
N_UTTERANCES = 30  # random subset of the FLEURS test set (865 utterances for Italian)
MODEL_SIZE = "small"  # "tiny", "base", "small", "medium", "large-v3-turbo", ...
BEAM_SIZE = 5  # 1 = greedy decoding
NORMALIZER = None  # None = language default; or "basic-v1", "whisper-basic", "it-v2", ...

DEVICE = "cuda" if ctranslate2.get_cuda_device_count() > 0 else "cpu"
print(f"Device: {DEVICE}")

## 3. Data

Downloads one FLEURS split from the Hugging Face Hub (the Italian test audio archive is about
660 MB, so this takes a minute or two), converts the selected utterances to 16 kHz mono WAV and
writes a JSONL manifest.

In [ ]:
from pathlib import Path

from asr_lab.data.ingest import ingest_fleurs
from asr_lab.data.manifest import read_manifest

dataset_dir = Path(f"data/fleurs_{LANG}_{N_UTTERANCES}")
manifest = ingest_fleurs(LANG, "test", dataset_dir, max_utterances=N_UTTERANCES)
entries = read_manifest(manifest)
print(f"{len(entries)} utterances, {sum(e.duration for e in entries) / 60:.1f} minutes of audio")
print(entries[0].text)

## 4. Transcribe

In [ ]:
from asr_lab.inference.faster_whisper import FasterWhisperModel
from asr_lab.inference.predictions import read_predictions
from asr_lab.inference.run import transcribe_manifest

model = FasterWhisperModel(MODEL_SIZE, device=DEVICE, beam_size=BEAM_SIZE)
run_dir = Path(f"runs/{model.name.replace('/', '-')}-beam{BEAM_SIZE}-fleurs-{LANG}-{N_UTTERANCES}")
predictions = transcribe_manifest(model, manifest, run_dir)

for entry, hypothesis in list(zip(entries, read_predictions(predictions), strict=True))[:3]:
    print("REF:", entry.text)
    print("HYP:", hypothesis.text)
    print()

## 5. Word error rate with a confidence interval

Corpus-level WER and CER after normalization, with a 95% bootstrap confidence interval. With only
a few dozen utterances the interval is wide: that is the point of reporting it.

In [ ]:
from asr_lab.evaluation.normalize import get_normalizer_by_name
from asr_lab.evaluation.scoring import format_summary, score_run

normalizer = get_normalizer_by_name(NORMALIZER) if NORMALIZER else None
scores, _ = score_run(run_dir, normalizer=normalizer)
print(format_summary(scores))

## 6. Where does the model go wrong?

In [ ]:
from asr_lab.evaluation.error_analysis import analyze_run, format_errors

utterances, report, _ = analyze_run(run_dir, normalizer=normalizer, top=10, worst=3)
print(format_errors(utterances, report, show=3))

## 7. The same transcripts, scored with different normalizers

Nothing is transcribed again: only the comparison between references and hypotheses changes.
See experiment E3 in RESULTS.md for the full analysis.

In [ ]:
names = ["none", "basic-v1", "whisper-basic", {"it": "it-v1", "de": "de-v1", "en": "en-v1"}[LANG]]
if LANG == "it":
    names.append("it-v2")
for name in names:
    scores, _ = score_run(run_dir, normalizer=get_normalizer_by_name(name))
    wer = scores["wer"]
    ci = wer["confidence_intervals"][wer["headline_unit"]]
    print(f"{name:14} WER {wer['estimate']:7.2%}  95% CI [{ci['low']:.2%}, {ci['high']:.2%}]")

## 8. Is the difference real? Paired comparison

Two normalizers (or two models) are compared on the same utterances with the paired bootstrap:
the interval of the difference excludes the noise of which utterances are in the sample.

In [ ]:
from asr_lab.evaluation.comparison import compare_runs, format_comparison

result = compare_runs(
    run_dir,
    run_dir,
    normalizer_a=get_normalizer_by_name("basic-v1"),
    normalizer_b=get_normalizer_by_name("whisper-basic"),
)
print(format_comparison(result))

## Next

- Change `MODEL_SIZE`, `BEAM_SIZE` or `LANG` in the settings cell and run everything again: each
  configuration gets its own run directory, so results are never mixed.
- To compare two models, transcribe with both and pass their run directories to `compare_runs`.
- The command line does the same: `asr-lab transcribe`, `score`, `errors`, `compare` (see the
  README).